# DATA ADQUISITION

In [52]:
import requests
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse, unquote

## Reading of API index files
We access the API at https://transparency-in-coverage.uhc.com to retrieve the total number of files and the total number of index files in the dataset.

In [54]:
API_URL = "https://transparency-in-coverage.uhc.com/api/v1/uhc/blobs/"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Referer": "https://transparency-in-coverage.uhc.com/",
})

response = session.get(API_URL, timeout=120)
#Check HTTPS response
response.raise_for_status()

blobs = response.json()["blobs"]

print("Total files returned by API:", len(blobs))

index_candidates = [
    b for b in blobs
    if b["name"].lower().endswith(
        ("_index.json")
    )
]

print("Files with 'index' in the name:", len(index_candidates))

for blob in index_candidates[:10]:
    print(blob["name"], "|", blob["size"])



Total files returned by API: 155551
Files with 'index' in the name: 68165
2026-10-01_1-800-RADIATOR-OF-DALLAS-FORT-WORTH-LLC_index.json | 2295
2026-10-01_1-AB-Inc_index.json | 2203
2026-10-01_1-AMERICAN-STEEL-WORKS_index.json | 1968
2026-10-01_1-BETTER-LLC_index.json | 1938
2026-10-01_1-IT-SOURCE-LLC_index.json | 2219
2026-10-01_1-REAL-ESTATE-MANAGEMENT-INC_index.json | 4636
2026-10-01_10-33-Ambulance-Service-LTD_index.json | 2255
2026-10-01_10-75-Emergency-Lighting-LLC_index.json | 3879
2026-10-01_10-8-VIDEO_index.json | 2204
2026-10-01_10-Huron-Condominium-Association_index.json | 2270


## Retrieving the First 1,000 Index Files

We access the API to retrieve the first 1,000 index files and download them to the local `uhc_data/indexes` directory.

We also create two log files in the same directory: `index_download_log.json`, which stores information about successfully downloaded index files, and `index_errors_log.json`, which records failed downloads along with the corresponding error messages.

In [6]:
BASE_DIR = Path("uhc_data")
INDEX_DIR = BASE_DIR / "indexes"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

index_blobs = [
    b for b in blobs
    if b["name"].lower().endswith(
        ("_index.json")
    )
]

# Mantener el orden recibido de la API
selected_indexes = index_blobs[:1000]

print("Total index files found:", len(index_blobs))
print("Index files selected for processing:", len(selected_indexes))

download_manifest = []
download_errors = []

for i, blob in enumerate(selected_indexes, start=1):
    url = blob["downloadUrl"]
    name = Path(blob["name"]).name
    destination = INDEX_DIR / name

    try:
        #Check if file already exists
        if not destination.exists():
            for attempt in range(3):
                try:
                  # GET file from API
                    with session.get(
                        url, stream=True, timeout=(20, 180)
                    ) as response:
                        #Check HTTPS response
                        response.raise_for_status()

                        #Write file in chunks to temporary file
                        with open(str(destination) + ".part", "wb") as f:
                            for chunk in response.iter_content(1024 * 1024):
                                if chunk:
                                    f.write(chunk)

                    # Rename temporary file to its final name
                    Path(str(destination) + ".part").replace(destination)
                    break

                except Exception:
                    # Remove temporary file
                    Path(str(destination) + ".part").unlink(
                        missing_ok=True
                    )
                    if attempt == 2:
                        raise

                    # Gives time in case server is temporary unavailable
                    time.sleep(2 ** attempt)

        download_manifest.append({
            "index_number": i,
            "name": name,
            "url": url,
            "local_path": str(destination),
            "status": "downloaded"
        })

    except Exception as e:
        download_errors.append({
            "name": name,
            "url": url,
            "error": str(e)
        })

    if i % 50 == 0:
        print(f"Index processed {i}/{len(selected_indexes)}")

# Save logs
(BASE_DIR / "index_download_log.json").write_text(
    json.dumps(download_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

(BASE_DIR / "index_errors_log.json").write_text(
    json.dumps(download_errors, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Correct downloads:", len(download_manifest))
print("Errors:", len(download_errors))

Total index files found: 68165
Index files selected for processing: 1000
Index processed 50/1000
Index processed 100/1000
Index processed 150/1000
Index processed 200/1000
Index processed 250/1000
Index processed 300/1000
Index processed 350/1000
Index processed 400/1000
Index processed 450/1000
Index processed 500/1000
Index processed 550/1000
Index processed 600/1000
Index processed 650/1000
Index processed 700/1000
Index processed 750/1000
Index processed 800/1000
Index processed 850/1000
Index processed 900/1000
Index processed 950/1000
Index processed 1000/1000
Correct downloads: 1000
Errors: 0


## Get size of the files from the downloaded index

### Funtion to get file size
Checks the size of a file by sending an HTTP HEAD request and returns its URL, size in bytes, and request status, handling errors when the size cannot be retrieved.

In [7]:
def get_file_size(url):
  try:
      response = session.head(
          url,
          allow_redirects=True,
          timeout=30
      )

      if response.status_code in (403, 405, 501):
          return {
              "url": url,
              "size_bytes": None,
              "status": f"HEAD no disponible: HTTP {response.status_code}"
          }

      response.raise_for_status()
      size = response.headers.get("Content-Length")

      return {
          "url": url,
          "size_bytes": int(size) if size else None,
          "status": "ok" if size else "Tamaño no indicado"
      }

  except Exception as e:
      return {
          "url": url,
          "size_bytes": None,
          "status": f"Error: {e}"
      }

### Calculating Total File Size and Identifying Files Associated with Index Files

We read the previously downloaded index files from the local `uhc_data/indexes` directory to identify the data files associated with each index. We then select the first 1,000 files referenced across the first 1,000 index files and retrieve their respective sizes.

The resulting information is stored in `uhc_data/indexes/files_associated.json`, including the relationship between each index file and its associated data files, as well as the size of each file.

We also calculate the total size of the 1,000 selected files, which amounts to approximately **595.165 GB**. This indicates that processing the entire dataset would require substantial storage and computing resources, so we will focus on a manageable subset of the data for this analysis.

In [13]:
BASE_DIR = Path("uhc_data")
INDEX_DIR = BASE_DIR / "indexes"

TARGET_FILES = 1000
MAX_WORKERS = 5


# Read locally stored index files
index_paths = sorted(INDEX_DIR.glob("*.json"))

print("Local index files found:", len(index_paths))

if not index_paths:
    raise FileNotFoundError(
        f"No index files were found in {INDEX_DIR.resolve()}"
    )


# Retrieve URLs while maintaining the relationship between indices and files
index_to_urls = {}
all_unique_urls = {}
used_indexes = []

for index_number, index_path in enumerate(index_paths, start=1):
    try:
        with open(index_path, "r", encoding="utf-8") as f:
            index_data = json.load(f)

        used_indexes.append(index_path.name)

        # Avoid duplicate URLs within each index
        index_urls = {}

        for structure in index_data.get("reporting_structure", []):

            # In-Network files
            for item in structure.get("in_network_files") or []:
                url = item.get("location")

                if url and url.startswith("https://"):
                    index_urls.setdefault(url, None)

            # Allowed Amount files (Out-of-Network)
            allowed = structure.get("allowed_amount_file") or {}
            url = allowed.get("location")

            if url and url.startswith("https://"):
                index_urls.setdefault(url, None)

        # Associate each index with its URLs
        index_to_urls[index_path.name] = list(index_urls.keys())

        # Collect all unique URLs across all indices
        for url in index_urls:
            all_unique_urls.setdefault(url, None)

    except Exception as e:
        print(f"Error processing {index_path.name}: {e}")

    if index_number % 100 == 0:
        print(
            f"Indices examined: {index_number}/{len(index_paths)} | "
            f"Unique files identified: {len(all_unique_urls)}"
        )


# Select the first 1,000 unique files
selected_urls = list(all_unique_urls.keys())[:TARGET_FILES]
selected_url_set = set(selected_urls)

print("\n--- Selection summary ---")
print("Indices processed:", len(used_indexes))
print("Unique files identified:", len(all_unique_urls))
print("Files selected:", len(selected_urls))


# Check file sizes in parallel
size_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(get_file_size, url)
        for url in selected_urls
    ]

    for i, future in enumerate(as_completed(futures), start=1):
        size_results.append(future.result())

        if i % 100 == 0 or i == len(selected_urls):
            print(
                f"File sizes checked: {i}/{len(selected_urls)}"
            )


# Calculate the total size of the selected files
total_bytes = sum(
    item["size_bytes"]
    for item in size_results
    if item["size_bytes"] is not None
)

known_sizes = sum(
    item["size_bytes"] is not None
    for item in size_results
)

unknown_sizes = len(size_results) - known_sizes

print("\n--- Total size of selected files ---")
print(f"Selected files: {len(selected_urls)}")
print(f"Files with known size: {known_sizes}")
print(f"Files with unknown size: {unknown_sizes}")
print(f"Total known size: {total_bytes / 1_000_000_000:.3f} GB")

if unknown_sizes > 0:
    print(
        "WARNING: The total size is partial because some files "
        "have an unknown size."
    )


# Create a dictionary to look up file size information by URL
size_by_url = {
    item["url"]: item
    for item in size_results
}


# Build the final dictionary mapping each index to its files and sizes
selection = {}

for index_name, urls in index_to_urls.items():

    index_files = []
    index_total_bytes = 0
    index_known_sizes = 0

    for url in urls:

        # Include only files in the global selection
        if url not in selected_url_set:
            continue

        size_info = size_by_url[url]
        size_bytes = size_info["size_bytes"]

        size_GB = (
            round(size_bytes / 1_000_000_000, 3)
            if size_bytes is not None
            else None
        )

        index_files.append({
            "url": url,
            "size_GB": size_GB,
            "status": size_info["status"]
        })

        if size_bytes is not None:
            index_total_bytes += size_bytes
            index_known_sizes += 1

    selection[index_name] = {
        "files": index_files,
        "total_files": len(index_files),
        "files_with_known_size": index_known_sizes,
        "files_with_unknown_size": len(index_files) - index_known_sizes,
        "total_known_GB": round(
            index_total_bytes / 1_000_000_000, 3
        )
    }


# Save the file selection and size information to a JSON file
selection_path = BASE_DIR / "files_associated.json"

with open(selection_path, "w", encoding="utf-8") as f:
    json.dump(selection, f, indent=2, ensure_ascii=False)

print("\n--- Final results ---")
print("Indices included:", len(selection))
print("Unique files selected:", len(selected_urls))
print(f"Total known size: {total_bytes / 1_000_000_000:.3f} GB")
print("JSON saved to:", selection_path.resolve())


Local index files found: 1000
Indices examined: 100/1000 | Unique files identified: 153
Indices examined: 200/1000 | Unique files identified: 279
Indices examined: 300/1000 | Unique files identified: 404
Indices examined: 400/1000 | Unique files identified: 540
Indices examined: 500/1000 | Unique files identified: 661
Indices examined: 600/1000 | Unique files identified: 785
Indices examined: 700/1000 | Unique files identified: 898
Indices examined: 800/1000 | Unique files identified: 1209
Indices examined: 900/1000 | Unique files identified: 1332
Indices examined: 1000/1000 | Unique files identified: 1453

--- Selection summary ---
Indices processed: 1000
Unique files identified: 1453
Files selected: 1000
File sizes checked: 100/1000
File sizes checked: 200/1000
File sizes checked: 300/1000
File sizes checked: 400/1000
File sizes checked: 500/1000
File sizes checked: 600/1000
File sizes checked: 700/1000
File sizes checked: 800/1000
File sizes checked: 900/1000
File sizes checked: 100

## Downloading Files from a Selected Index

Since the total dataset size is too large to process with the available computing resources, we decided to download the files associated with a single index file: `2026-10-01_10-33-Ambulance-Service-LTD_index.json`.

The selected index references six files: five In-Network pricing files and one Allowed Amounts file. We downloaded all six files and recorded their estimated and actual download sizes.

All six downloads completed successfully, with no failed downloads. The total downloaded size was approximately **18.304 GB**.

The files were saved to `/content/uhc_data/associated_files/index_files`, and the download log was stored in `files_download_log.json` in the same directory.

Although all six files were downloaded successfully, their sizes vary considerably. Two In-Network files account for approximately 18.3 GB combined, making them particularly demanding to process with the available computing resources. We will therefore assess the files individually and select a manageable subset for further analysis.

In [17]:
# 1. Load the file selection generated previously
SELECTION_PATH = Path("/content/uhc_data/files_associated.json")

# Use the exact index filename as it appears in files_associated.json
INDEX_NAME = "2026-10-01_10-33-Ambulance-Service-LTD_index.json"

# 2. Output directory
OUTPUT_DIR = Path("/content/uhc_data/associated_files/index_files")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Load the selected files for this index
with open(SELECTION_PATH, "r", encoding="utf-8") as f:
    selection = json.load(f)

if INDEX_NAME not in selection:
    raise ValueError(f"Index not found in selection: {INDEX_NAME}")

files = selection[INDEX_NAME]["files"]

# Only include files with a valid URL and successful size check
files_to_download = [
    item for item in files
    if item.get("url") and item.get("status") == "ok"
]

print("Index:", INDEX_NAME)
print("Files available for download:", len(files_to_download))

results = []

for i, item in enumerate(files_to_download, start=1):
    url = item["url"]
    filename = unquote(Path(urlparse(url).path).name)
    destination = OUTPUT_DIR / filename

    try:
        print(f"\n[{i}/{len(files_to_download)}] Downloading: {filename}")
        print(f"Estimated size: {item.get('size_GB', 'unknown')} GB")

        with session.get(
            url,
            stream=True,
            allow_redirects=True,
            timeout=(20, 180)
        ) as response:
            response.raise_for_status()

            bytes_downloaded = 0

            with open(destination, "wb") as f:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)
                        bytes_downloaded += len(chunk)

        print(f"Downloaded: {bytes_downloaded / 1e9:.3f} GB")

        results.append({
            "url": url,
            "filename": filename,
            "status": "downloaded",
            "size_bytes": bytes_downloaded,
        })

    except Exception as e:
        if destination.exists():
            destination.unlink()

        print(f"ERROR: {e}")

        results.append({
            "url": url,
            "filename": filename,
            "status": "error",
            "error": str(e),
        })

# 5. Save the download manifest
manifest_path = OUTPUT_DIR / "files_download_log.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

successful = sum(
    item["status"] == "downloaded" for item in results
)
failed = len(results) - successful

total_bytes = sum(
    item.get("size_bytes", 0)
    for item in results
    if item["status"] == "downloaded"
)

print("\n--- DOWNLOAD SUMMARY ---")
print("Files selected:", len(files_to_download))
print("Successfully downloaded:", successful)
print("Failed downloads:", failed)
print(f"Total downloaded: {total_bytes / 1e9:.3f} GB")
print("Output directory:", OUTPUT_DIR)
print("Download log file:", manifest_path)


Index: 2026-10-01_10-33-Ambulance-Service-LTD_index.json
Files available for download: 6

[1/6] Downloading: 2026-10-01_United-HealthCare-Services--Inc-_Third-Party-Administrator_Choice-Plus_8_in-network-rates.json.gz
Estimated size: 9.434 GB
Downloaded: 9.434 GB

[2/6] Downloading: 2026-10-01_United-HealthCare-Services--Inc-_Third-Party-Administrator_Core_579_in-network-rates.json.gz
Estimated size: 8.864 GB
Downloaded: 8.864 GB

[3/6] Downloading: 2026-10-01_United-HealthCare-Services--Inc-_Third-Party-Administrator_OHPH-Acupuncture-Massage-Naturopath_31_in-network-rates.json.gz
Estimated size: 0.0 GB
Downloaded: 0.000 GB

[4/6] Downloading: 2026-10-01_United-HealthCare-Services--Inc-_Third-Party-Administrator_OHPH-Chiro_28_in-network-rates.json.gz
Estimated size: 0.006 GB
Downloaded: 0.006 GB

[5/6] Downloading: 2026-10-01_United-HealthCare-Services--Inc-_Third-Party-Administrator_OHPH-ST_30_in-network-rates.json.gz
Estimated size: 0.001 GB
Downloaded: 0.001 GB

[6/6] Downloading: 2

# DATA PROCESSING

In [18]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
%pip install -q pyspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.20.1+1-1~24.04) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.020s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.020s

In [39]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, ArrayType

In [40]:

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = (
    SparkSession.builder
    .appName("OptumUHCDataAssessment")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


## Reading Allowed Amounts

After inspecting the `allowed_amounts` data, we found no out-of-network pricing information available in the selected data. Therefore, we will focus our analysis on a single In-Network pricing table.

This allows us to concentrate on the available negotiated rates while keeping the scope of the analysis aligned with the data successfully retrieved and processed.

In [41]:
allowed_amount = (
    spark.read
    .option("multiLine", True)
    .json("/content/uhc_data/associated_files/index_files/2026-10-01_10-33-Ambulance-Service-LTD_United-Healthcare-POS_allowed-amounts.json.gz")
)

allowed_amount.printSchema()

root
 |-- last_updated_on: string (nullable = true)
 |-- out_of_network: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- reporting_entity_name: string (nullable = true)
 |-- reporting_entity_type: string (nullable = true)
 |-- sourceSystem_plan: string (nullable = true)
 |-- version: string (nullable = true)



In [42]:
allowed_amount.show()

+---------------+--------------+---------------------+---------------------+-----------------+-------+
|last_updated_on|out_of_network|reporting_entity_name|reporting_entity_type|sourceSystem_plan|version|
+---------------+--------------+---------------------+---------------------+-----------------+-------+
|     2026-10-01|            []| United-HealthCare...| Third-Party Admin...|           Cirrus|  2.0.0|
+---------------+--------------+---------------------+---------------------+-----------------+-------+



## Reading In-Network Data

The selected index file references five In-Network files. Since all five files share the same schema, we selected one file for processing.

Some of the larger files, approximately 9 GB each, were excluded because they could not be processed successfully with the available computing resources.

This approach allowed us to work with a manageable subset of the data while preserving the expected schema. However, the findings are based on the selected file and may not be representative of the entire dataset.

In [43]:
acupuncture_df = (
    spark.read
    .option("multiLine", True)
    .json("/content/uhc_data/associated_files/index_files/2026-10-01_United-HealthCare-Services--Inc-_Third-Party-Administrator_OHPH-Acupuncture-Massage-Naturopath_31_in-network-rates.json.gz")
)

acupuncture_df.printSchema()

root
 |-- in_network: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- billing_code: string (nullable = true)
 |    |    |-- billing_code_type: string (nullable = true)
 |    |    |-- billing_code_type_version: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- negotiated_rates: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- negotiated_prices: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- additional_information: string (nullable = true)
 |    |    |    |    |    |    |-- billing_class: string (nullable = true)
 |    |    |    |    |    |    |-- billing_code_modifier: array (nullable = true)
 |    |    |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |    |    |-- expiration_date: strin

## Flattening JSON Data

Since the source data is stored in JSON format, we flatten the nested structures to make the data easier to access and analyze in a tabular format.

This process produces two tables:

- **In-network (pricing) table:** Contains information about the negotiated rates for the different healthcare services.
- **Provider references table:** Contains information about the providers associated with the services, referenced by their identifiers.

Once both tables have been created, we check for duplicate records in each table before joining them. This helps prevent unintended row multiplication during the join and ensures that the resulting dataset preserves the expected data granularity.

Finally, we join the two tables using the relevant provider reference identifiers to associate each negotiated rate with its corresponding provider information.

In [44]:
# PROCESS IN_NETWORK PRICES
def flatten_in_network_prices(df):

    return (
        df
        .select(
            "reporting_entity_name",
            "reporting_entity_type",
            "last_updated_on",
            "version",
            F.explode_outer("in_network").alias("item")
        )
        .select(
            "reporting_entity_name",
            "reporting_entity_type",
            "last_updated_on",
            "version",
            "item.*"
        )
        .withColumn(
            "rate",
            F.explode_outer("negotiated_rates")
        )
        .drop("negotiated_rates")
        .select("*", "rate.*")
        .drop("rate")
        .withColumn(
            "price",
            F.explode_outer("negotiated_prices")
        )
        .drop("negotiated_prices")
        .select("*", "price.*")
        .drop("price")
        .withColumn(
            "provider_reference_id",
            F.explode_outer("provider_references")
        )
        .drop("provider_references")
        .withColumn(
            "service_code",
            F.explode_outer("service_code")
        )
        .withColumn(
            "billing_code_modifier",
            F.explode_outer("billing_code_modifier")
        )
    )

#PROCESS PROVIDER_REFERENCESç
def flatten_providers(df):

    return (
        df
        .select(
            F.explode_outer("provider_references").alias("pref")
        )
        .select("pref.*")
        .withColumn(
            "provider_group",
            F.explode_outer("provider_groups")
        )
        .drop("provider_groups")
        .select(
            "*",
            F.col("provider_group.tin").alias("tin"),
            F.explode_outer("provider_group.npi").alias("npi")
        )
        .drop("provider_group")
        .select(
            "provider_group_id",
            F.explode_outer("network_name").alias("network_name"),
            F.col("tin.business_name").alias("tin_business_name"),
            F.col("tin.type").alias("tin_type"),
            F.col("tin.value").alias("tin_value"),
            "npi"
        )
    )

# JOIN PRICES WITH PROVIDERS
def flatten_uhc_file(df):

    prices = flatten_in_network_prices(df)
    providers = flatten_providers(df)

    print("Original rows in prices:", prices.count())
    print("Unique rows in prices", prices.dropDuplicates().count())

    print("Original rows in providers:", providers.count())
    print("Unique rows in providers", providers.dropDuplicates().count())

    prices = prices.dropDuplicates()
    providers = providers.dropDuplicates()

    result = (
        prices.alias("p")
        .join(
            providers.alias("v"),
            F.col("p.provider_reference_id")
            == F.col("v.provider_group_id"),
            "left"
        )
        .select(
            "p.*",
            F.col("v.network_name").alias("network_name"),
            F.col("v.tin_business_name").alias("tin_business_name"),
            F.col("v.tin_type").alias("tin_type"),
            F.col("v.tin_value").alias("tin_value"),
            F.col("v.npi").alias("npi")
        )
    )

    return result

In [45]:
flat_df = flatten_uhc_file(acupuncture_df)

flat_df.printSchema()
flat_df.show(20, truncate=False)


Original rows in prices: 27838
Unique rows in prices 27838
Original rows in providers: 5096
Unique rows in providers 5092
root
 |-- reporting_entity_name: string (nullable = true)
 |-- reporting_entity_type: string (nullable = true)
 |-- last_updated_on: string (nullable = true)
 |-- version: string (nullable = true)
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_code_type_version: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- severity_of_illness: string (nullable = true)
 |-- additional_information: string (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: string (nullable = true)
 |-- expiration_date: string (nullable = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: string (nullable = true)
 |-- sett

# DATA ANALYSIS

## Data Quality

We assessed the data quality of each column in the table by collecting the following metrics:

- Data type
- Null count
- Blank count
- Null percentage
- Blank percentage
- Completeness percentage

The analysis shows that the `severity_of_illness` and `additional_information` columns contain only blank values. Since these fields may be optional, this could indicate that no additional information was provided in the selected dataset. However, this assumption would need to be confirmed against the source data specifications.

Another column of interest is `billing_code_modifier`, which appears to contain codes that provide additional context for the corresponding `billing_code`. Its completeness is only 0.18%, suggesting that modifiers are present in a very small proportion of records. This may be expected if modifiers are only required for specific services or billing scenarios.




In [46]:
def check_quality(df):
    total_rows = df.count()
    columns = df.columns

    #Check nulls and blank values
    expressions = []

    for field in df.schema.fields:
        c = field.name
        col = F.col(f"`{c}`")

        expressions.append(
            F.sum(
                F.when(col.isNull(), 1).otherwise(0)
            ).alias(f"null__{c}")
        )

        if isinstance(field.dataType, StringType):
            expressions.append(
                F.sum(
                    F.when(
                        col.isNotNull() & (F.trim(col) == ""),
                        1
                    ).otherwise(0)
                ).alias(f"blank__{c}")
            )

    result = df.agg(*expressions).first().asDict()

    # Construir una tabla de resultados
    rows = []

    for field in df.schema.fields:
        c = field.name
        nulls = result.get(f"null__{c}") or 0
        blanks = result.get(f"blank__{c}", 0) or 0

        missing = nulls + blanks

        rows.append((
            c,
            field.dataType.simpleString(),
            int(nulls),
            int(blanks),
            round(100 * nulls / total_rows, 2) if total_rows else 0,
            round(100 * missing / total_rows, 2) if total_rows else 0,
            round(100 * (total_rows - missing) / total_rows, 2)
            if total_rows else 0
        ))

    profile_schema = [
        "column",
        "data_type",
        "null_count",
        "blank_count",
        "null_pct",
        "missing_pct",
        "completeness_pct"
    ]

    profile_df = spark.createDataFrame(rows, profile_schema)

    print(f"Total de filas: {total_rows}")
    print(f"Total de columnas: {len(columns)}")

    profile_df.orderBy(
        F.desc("missing_pct")
    ).show(len(columns), truncate=False)

    return profile_df


quality_profile = check_quality(flat_df)




Total de filas: 467861
Total de columnas: 25
+-------------------------+---------+----------+-----------+--------+-----------+----------------+
|column                   |data_type|null_count|blank_count|null_pct|missing_pct|completeness_pct|
+-------------------------+---------+----------+-----------+--------+-----------+----------------+
|severity_of_illness      |string   |0         |467861     |0.0     |100.0      |0.0             |
|additional_information   |string   |0         |467861     |0.0     |100.0      |0.0             |
|billing_code_modifier    |string   |467019    |0          |99.82   |99.82      |0.18            |
|reporting_entity_name    |string   |0         |0          |0.0     |0.0        |100.0           |
|billing_class            |string   |0         |0          |0.0     |0.0        |100.0           |
|reporting_entity_type    |string   |0         |0          |0.0     |0.0        |100.0           |
|expiration_date          |string   |0         |0          |0.0 

In [47]:
flat_df.select("billing_code_modifier", "billing_code").filter("billing_code_modifier = '51'").show()

+---------------------+------------+
|billing_code_modifier|billing_code|
+---------------------+------------+
|                   51|       98943|
|                   51|       98943|
|                   51|       98943|
+---------------------+------------+



## Duplicate Check

Initially, 1,900 duplicate records were identified. After investigating the potential causes, the issue was traced to the join operation.

We verified that the pricing table contained no duplicate records, while the providers table contained four duplicate records.

After removing these duplicates from the providers table, no duplicate records remained in the joined dataset. The initial duplicates were caused by the join operation, as each provider could be associated with multiple pricing records.

In [48]:
# Duplicates considering all columns
total_rows = flat_df.count()
distinct_rows = flat_df.dropDuplicates().count()

print("Total de filas:", total_rows)
print("Filas distintas:", distinct_rows)
print("Duplicados exactos:", total_rows - distinct_rows)


Total de filas: 467861
Filas distintas: 467861
Duplicados exactos: 0


## Pricing Data Quality and Overview

We performed additional data quality checks on the negotiated pricing data to assess price validity, pricing types, billing classification and dataset coverage.

- **Price validity:** The dataset contains 467,861 records, with no null, negative or zero prices. The minimum price is 0.14 and the maximum is 6,615.00. This indicates that all records have a positive, non-null price value.

- **Negotiated type:** Almost all records (467,852, approximately 99.998%) are classified as `negotiated`, while only 9 records are classified as `per diem`. This indicates that the dataset is overwhelmingly composed of negotiated rates, with a very small number of per-diem records.

- **Billing classification:** All 467,861 records are classified as `professional`. Therefore, the analyzed data covers professional billing records only, and no other billing classifications are represented in this dataset.

- **Dataset coverage:** The dataset contains 3,149 distinct billing codes, 160 distinct provider reference IDs and 4,891 distinct NPIs. It suggests that a provider reference may be associated with multiple NPIs. Also, this demonstrates that the selected file includes a wide variety of healthcare services and multiple provider identifiers. The number of distinct billing codes also indicates that the analysis covers many different types of services rather than a small set of procedures.

Overall, the initial data quality checks show no missing or negative prices, and the dataset has broad coverage across billing codes and NPIs.

In [49]:
# Prices quality
flat_df.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.col("negotiated_rate").isNull(), 1).otherwise(0)
    ).alias("null_prices"),
    F.sum(
        F.when(F.col("negotiated_rate") < 0, 1).otherwise(0)
    ).alias("negative_prices"),
    F.sum(
        F.when(F.col("negotiated_rate") == 0, 1).otherwise(0)
    ).alias("zero_prices"),
    F.min("negotiated_rate").alias("min_price"),
    F.max("negotiated_rate").alias("max_price")
).show(truncate=False)

# Negotiation types distribution
flat_df.groupBy("negotiated_type").count().orderBy(
    F.desc("count")
).show(truncate=False)

# Billing class distribution
flat_df.groupBy("billing_class").count().orderBy(
    F.desc("count")
).show(truncate=False)

# Principal identifiers
flat_df.select(
    F.countDistinct("billing_code").alias("distinct_billing_codes"),
    F.countDistinct("provider_reference_id").alias("distinct_provider_references"),
    F.countDistinct("npi").alias("distinct_npis")
).show()


+----------+-----------+---------------+-----------+---------+---------+
|total_rows|null_prices|negative_prices|zero_prices|min_price|max_price|
+----------+-----------+---------------+-----------+---------+---------+
|467861    |0          |0              |0          |0.14     |6615.0   |
+----------+-----------+---------------+-----------+---------+---------+

+---------------+------+
|negotiated_type|count |
+---------------+------+
|negotiated     |467852|
|per diem       |9     |
+---------------+------+

+-------------+------+
|billing_class|count |
+-------------+------+
|professional |467861|
+-------------+------+

+----------------------+----------------------------+-------------+
|distinct_billing_codes|distinct_provider_references|distinct_npis|
+----------------------+----------------------------+-------------+
|                  3149|                         160|         4891|
+----------------------+----------------------------+-------------+



## Pricing Analysis — Key Findings

The pricing analysis reveals several interesting patterns in the distribution of negotiated rates across healthcare services.

- **Uniform pricing for certain billing codes:** Several services have identical minimum, median and maximum prices, with a standard deviation of zero. For example, billing codes `86769` and `86328` have fixed prices of 42.13 and 45.23, respectively, across 5,084 records and 4,891 distinct NPIs. This suggests consistent pricing for these services within the analyzed dataset.

- **Significant price variability for other services:** Some billing codes exhibit substantial differences between their minimum and maximum negotiated rates. For example, code `99204` (new office/outpatient visit, moderate complexity) ranges from 48 to 113, while code `97140` (manual therapy) ranges from 13 to 40. These differences indicate that negotiated rates are not equally consistent across all services.

- **Differences in provider coverage:** The number of distinct NPIs varies by billing code. The most frequent codes in the results have 5,084 records and 4,891 distinct NPIs, while other codes have 3,363 records and 3,231 distinct NPIs. This shows that the number of providers associated with each service differs across the analyzed data.

- **Acupuncture services show price variability:** Billing codes `97810`, `97811`, `97813` and `97814` have multiple distinct negotiated rates. For example, code `97813` ranges from 16.50 to 57, with a median of 40. This highlights differences in negotiated rates for these services within the sample.

- **Price dispersion is not always reflected by the median:** For code `97035`, the median price is 10, while the maximum reaches 29.80. Similarly, code `97140` has a median of 13 and a maximum of 40. These examples show why examining only the median may not fully capture the range of negotiated rates.

- **The dataset contains services with different levels of representation:** The number of records per billing code varies considerably, from more than 5,000 records for some codes to fewer than 200 for others in the displayed results. Therefore, the amount of evidence available for each service differs, and this should be considered when interpreting the findings.

Overall, the results suggest that negotiated pricing patterns vary by billing code: some services exhibit uniform rates, while others show substantial price dispersion.

These findings are based on the selected In-Network file and the billing codes displayed in the analysis.

In [50]:
# Prices distribution across billing code
price_by_code = (
    flat_df
    .groupBy(
        "billing_code",
        "billing_code_type",
        "name"
    )
    .agg(
        # Volume and provider coverage
        F.count("*").alias("row_count"),
        F.countDistinct("npi").alias("distinct_npis"),

        # Price distribution
        F.countDistinct("negotiated_rate").alias("distinct_prices"),
        F.min("negotiated_rate").alias("min_price"),
        F.expr(
            "percentile_approx(negotiated_rate, 0.25)"
        ).alias("p25_price"),
        F.expr(
            "percentile_approx(negotiated_rate, 0.5)"
        ).alias("median_price"),
        F.expr(
            "percentile_approx(negotiated_rate, 0.75)"
        ).alias("p75_price"),
        F.max("negotiated_rate").alias("max_price"),

        # Price dispersion
        F.stddev("negotiated_rate").alias("stddev_price"),
        F.avg("negotiated_rate").alias("avg_price")
    )
    # Derived metrics
    .withColumn(
        "price_range_ratio",
        F.when(
            F.col("min_price") > 0,
            F.col("max_price") / F.col("min_price")
        )
    )
    .withColumn(
        "coefficient_of_variation",
        F.when(
            F.col("avg_price") > 0,
            F.col("stddev_price") / F.col("avg_price")
        )
    )
    .withColumn(
        "price_variability",
        F.when(
            F.col("distinct_prices") == 1,
            F.lit("Uniform pricing")
        )
        .when(
            F.col("coefficient_of_variation") <= 0.10,
            F.lit("Low variability")
        )
        .otherwise(
            F.lit("Higher variability")
        )
    )
)

price_by_code.orderBy(
    F.desc("row_count")
).show(50, truncate=False)

+------------+-----------------+------------------------------------------------+---------+-------------+---------------+---------+---------+------------+---------+---------+---------------------+------------------+------------------+------------------------+------------------+
|billing_code|billing_code_type|name                                            |row_count|distinct_npis|distinct_prices|min_price|p25_price|median_price|p75_price|max_price|stddev_price         |avg_price         |price_range_ratio |coefficient_of_variation|price_variability |
+------------+-----------------+------------------------------------------------+---------+-------------+---------------+---------+---------+------------+---------+---------+---------------------+------------------+------------------+------------------------+------------------+
|86769       |CPT              |ANTB SEVERE AQT RESPIR SYND SARS-COV-2 COVID-19 |5084     |4891         |1              |42.13    |42.13    |42.13       |42.13    

## Provider Join Validation

We performed a `LEFT JOIN` between the In-Network pricing table and the provider reference table using `provider_reference_id`, preserving all records from the pricing table.

Since this is a left join, the pricing records are preserved even when no matching provider record exists. Therefore, the absence of null NPIs suggests complete matching for this field in the resulting dataset.

In [51]:
flat_df.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(
            F.col("provider_reference_id").isNotNull(),
            1
        ).otherwise(0)
    ).alias("rows_with_reference"),
    F.sum(
        F.when(
            F.col("provider_reference_id").isNotNull()
            & F.col("npi").isNull(),
            1
        ).otherwise(0)
    ).alias("references_without_npi")
).show(truncate=False)


+----------+-------------------+----------------------+
|total_rows|rows_with_reference|references_without_npi|
+----------+-------------------+----------------------+
|467861    |467861             |0                     |
+----------+-------------------+----------------------+



# WRITE DATA
Save the flattened DataFrame in Parquet format for subsequent upload to Snowflake.

In [56]:
OUTPUT_DIR = "/content/uhc_data/output/flat_df_parquet"

# Create parent directory
Path(OUTPUT_DIR).parent.mkdir(parents=True, exist_ok=True)

# Count rows before exporting
row_count = flat_df.count()

# Write a single Parquet file
flat_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(OUTPUT_DIR)

print("Parquet export completed successfully")
print(f"Rows exported: {row_count}")
print(f"Output directory: {OUTPUT_DIR}")

Parquet export completed successfully
Rows exported: 467861
Output directory: /content/uhc_data/output/flat_df_parquet
